# 🌑 Crater Detection — Exploratory Data Analysis

Dataset: **juliensimon/lunar-craters-robbins** — 384,278 real craters (Robbins 2019 / USGS)

This notebook covers:
1. Loading and exploring the Robbins catalog
2. Real class distribution + imbalance analysis
3. Robbins-seeded tile generation
4. Comparison: Robbins tiles vs pure synthetic
5. IoT telemetry simulation demo

In [ ]:
import sys
sys.path.insert(0, '../src')

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

from utils import CLASS_NAMES, CLASS_COLORS, draw_detections, PROJECT_ROOT
from data_pipeline import (
    load_robbins_catalog, sample_catalog_region,
    craters_to_tile_annotations, render_tile_from_annotations,
    generate_synthetic_image, KM_PER_TILE
)
from iot_simulator import InProcessBroker, EdgeNode, GroundStation

plt.rcParams['figure.facecolor'] = '#0d1117'
plt.rcParams['axes.facecolor']   = '#1a1a2e'
plt.rcParams['text.color']       = 'white'
plt.rcParams['axes.labelcolor']  = 'white'
plt.rcParams['xtick.color']      = 'white'
plt.rcParams['ytick.color']      = 'white'

print('Setup complete ✅')

## 1. Load and Explore the Robbins Catalog

In [ ]:
# Load real Robbins catalog (4.3 MB, cached after first download)
df = load_robbins_catalog()
print(df.head())
print(f'\nShape: {df.shape}')
print(f'\nColumns: {list(df.columns)}')
print(f'\nSize class distribution:')
print(df['size_class'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Robbins Lunar Crater Catalog — 384,278 craters', fontsize=13)

# Diameter distribution (log scale)
axes[0].hist(df['diameter_km'], bins=100, log=True,
             color='#58a6ff', edgecolor='#0d1117', alpha=0.85)
axes[0].set_xlabel('Diameter (km)')
axes[0].set_ylabel('Count (log scale)')
axes[0].set_title('Diameter Distribution')

# Class distribution bar
counts = df['size_class'].value_counts()
colors = ['#e74c3c', '#2ecc71', '#3498db', '#9b59b6']
bars = axes[1].bar(counts.index, counts.values,
                   color=colors[:len(counts)], edgecolor='#0d1117')
axes[1].bar_label(bars, padding=3, fmt='%d')
axes[1].set_title('Class Distribution')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=15)

# Spatial density — global crater map
sample = df.sample(5000, random_state=42)
sc = axes[2].scatter(
    sample['longitude_deg'], sample['latitude_deg'],
    c=np.log10(sample['diameter_km']),
    cmap='plasma', s=1, alpha=0.5
)
plt.colorbar(sc, ax=axes[2], label='log10(diameter km)')
axes[2].set_xlabel('Longitude (°E)')
axes[2].set_ylabel('Latitude (°)')
axes[2].set_title('Global Crater Map (5k sample)')

plt.tight_layout()
plt.show()

print('\n→ Strong class imbalance: small craters dominate 87.6% of catalog')
print('→ Mitigation: oversampling large-crater regions + class weights in loss')

## 2. Robbins-Seeded Tile Generation
Sample a real crater cluster, project positions onto a 256×256 tile, render terrain.

In [ ]:
# Pick 8 interesting regions from the catalog
seeds = df[df['size_class'].isin(['large','giant'])].sample(8, random_state=7)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Robbins-Seeded Lunar Tiles (real crater positions + sizes)', fontsize=13)

for ax, (_, seed) in zip(axes.flat, seeds.iterrows()):
    clat = float(seed['latitude_deg'])
    clon = float(seed['longitude_deg'])

    region = sample_catalog_region(df, clat, clon, KM_PER_TILE)
    anns   = craters_to_tile_annotations(region, clat, clon, KM_PER_TILE)
    img    = render_tile_from_annotations(anns)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

    # Draw boxes
    boxes, classes = [], []
    for a in anns:
        cx, cy, w, h = a['cx']*256, a['cy']*256, a['w']*256, a['h']*256
        boxes.append((int(cx-w/2), int(cy-h/2), int(cx+w/2), int(cy+h/2)))
        classes.append(a['class'])

    annotated = draw_detections(img_rgb, boxes, classes)
    ax.imshow(annotated)
    ax.set_title(f'Lat:{clat:.1f}° Lon:{clon:.1f}°\n{len(anns)} craters', fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()

## 3. Robbins vs Synthetic — Side-by-Side

In [ ]:
import random
random.seed(42)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Top row: Robbins-seeded  |  Bottom row: Pure synthetic', fontsize=12)

# Top: Robbins
seeds2 = df.sample(4, random_state=99)
for ax, (_, seed) in zip(axes[0], seeds2.iterrows()):
    clat, clon = float(seed['latitude_deg']), float(seed['longitude_deg'])
    region = sample_catalog_region(df, clat, clon, KM_PER_TILE)
    anns   = craters_to_tile_annotations(region, clat, clon, KM_PER_TILE)
    img    = render_tile_from_annotations(anns)
    ax.imshow(img, cmap='gray')
    ax.set_title(f'Robbins: {len(anns)} craters', fontsize=9)
    ax.axis('off')

# Bottom: Synthetic
for ax in axes[1]:
    img, anns = generate_synthetic_image(256, 256)
    ax.imshow(img, cmap='gray')
    ax.set_title(f'Synthetic: {len(anns)} craters', fontsize=9)
    ax.axis('off')

plt.tight_layout()
plt.show()

print('Robbins tiles: crater sizes/positions reflect real lunar geology')
print('Synthetic tiles: uniform random — good fallback, less realistic')

## 4. Class Imbalance — Real vs Synthetic Distribution

In [ ]:
# Real distribution from Robbins
real_counts = df['size_class'].value_counts()
# Map giant → large for 3-class setup
real_3class = {
    'small':  int(real_counts.get('small', 0)),
    'medium': int(real_counts.get('medium', 0)),
    'large':  int(real_counts.get('large', 0)) + int(real_counts.get('giant', 0)),
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Class Imbalance Analysis', fontsize=13)

colors = ['#e74c3c', '#2ecc71', '#3498db']

# Real counts (log scale to show imbalance)
bars = ax1.bar(real_3class.keys(), real_3class.values(),
               color=colors, edgecolor='#0d1117')
ax1.bar_label(bars, padding=3, fmt='%d')
ax1.set_title('Robbins Catalog (384k craters)')
ax1.set_ylabel('Count')
ax1.set_yscale('log')

# Percentage pie
total = sum(real_3class.values())
pcts  = [v/total*100 for v in real_3class.values()]
ax2.pie(pcts, labels=[f"{k}\n{p:.1f}%" for k,p in zip(real_3class.keys(), pcts)],
        colors=colors, startangle=90,
        textprops={'color': 'white', 'fontsize': 11})
ax2.set_title('Real Lunar Size Distribution')

plt.tight_layout()
plt.show()

print('Imbalance ratio small:medium:large ≈',
      f"{real_3class['small']//real_3class['medium']}:1:"
      f"{real_3class['medium']//real_3class['large']}")
print('→ Mitigation in pipeline:')
print('   1. Oversample large-crater tiles (2× in training split)')
print('   2. Class weights [2.5, 1.0, 0.8] in YOLO loss')
print('   3. Lower confidence threshold (0.25) for small crater detection')

## 5. IoT Telemetry Simulation Demo

In [ ]:
import time
from iot_simulator import InProcessBroker, EdgeNode, GroundStation

# Set up in-process broker + edge + ground station
broker  = InProcessBroker()
station = GroundStation(broker)
node    = EdgeNode(broker, target='Moon', interval_sec=0.3)

print('Emitting 10 synthetic telemetry events from edge node …\n')
for i in range(10):
    p = node.emit_once()
    alert = ' ⚠ HIGH DENSITY' if p.high_density_flag else ''
    anom  = ' 🔴 ANOMALY' if p.anomaly_flag else ''
    print(f'[{i+1:2d}] tile={p.tile_id}  craters={p.n_craters_detected:2d}  '
          f'conf={p.avg_confidence:.2f}  alt={p.altitude_km:.0f}km  '
          f'temp={p.sensor_temp_c:.0f}°C{alert}{anom}')

print()
station.print_dashboard()

In [ ]:
# Plot telemetry time series
series = station.get_telemetry_series()

fig, axes = plt.subplots(2, 2, figsize=(12, 6))
fig.suptitle('IoT Telemetry — Edge Node Output', fontsize=13)

# Detection counts
axes[0,0].plot(series['crater_counts'], '-o', color='#58a6ff', ms=5)
axes[0,0].axhline(12, ls='--', color='#f85149', alpha=0.7, label='Alert threshold')
axes[0,0].set_title('Crater Count per Tile')
axes[0,0].set_ylabel('Count')
axes[0,0].legend()

# Confidence
axes[0,1].plot(series['confidences'], '-o', color='#2ecc71', ms=5)
axes[0,1].set_title('Average Detection Confidence')
axes[0,1].set_ylim(0, 1)

# Altitude
axes[1,0].plot(series['altitudes_km'], '-o', color='#f39c12', ms=5)
axes[1,0].set_title('Orbital Altitude (km)')
axes[1,0].set_ylabel('km')

# Temperature
axes[1,1].plot(series['temps_c'], '-o', color='#e74c3c', ms=5)
axes[1,1].axhline(110, ls='--', color='#f85149', alpha=0.7, label='Thermal limit')
axes[1,1].set_title('Sensor Temperature (°C)')
axes[1,1].set_ylabel('°C')
axes[1,1].legend()

plt.tight_layout()
plt.show()

## 6. Next Steps

```bash
# 1. Generate dataset from Robbins catalog (recommended)
python src/data_pipeline.py --robbins --n 600

# OR: no-internet fallback
python src/data_pipeline.py --synthetic --n 600

# 2. Train both models
python src/train.py --all --epochs 50

# 3. Evaluate
python src/evaluate.py --all

# 4. Visualise
python src/visualize.py --model yolov8n --demo

# 5. Launch app
streamlit run app/app.py
```